# Customer Churn Prediction & Retention Analytics — Walkthrough

This notebook narrates the project end-to-end and is meant to be read top-to-bottom. It reuses the **same production modules** in `src/` (no copy-pasted logic), so what you see here is exactly what `python main.py` runs.

**Storyline**
1. Load & clean data, engineer features
2. Statistical EDA (what *significantly* relates to churn?)
3. Train, calibrate, evaluate a model bake-off
4. Turn probabilities into **profit** (cost-sensitive threshold)
5. Survival analysis — *when* do customers leave?
6. CLV & **Expected Value at Risk** — *which* churners cost the most?
7. Segmentation & uplift — *who* should we actually contact?

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import pandas as pd
import numpy as np
pd.set_option('display.max_columns', 50)

from src.utils import load_config, set_global_seed
config = load_config('../config.yaml')
set_global_seed(config['project']['random_seed'])
TARGET = config['data']['target_column']
config['project']['name']

'Customer Churn Prediction & Retention Analytics'

## 1. Data + feature engineering
`make_dataset` downloads the real IBM Telco data (or builds a high-fidelity synthetic fallback). `clean_data` fixes the well-known blank `TotalCharges`, and `add_engineered_features` adds domain features (service counts, tenure buckets, autopay flags, ...).

In [2]:
from src.data.make_dataset import get_raw_data
from src.data.preprocess import clean_data
from src.features.build_features import add_engineered_features

raw = get_raw_data(config)
cleaned = clean_data(raw, config)
featured = add_engineered_features(cleaned)
print(f'Rows: {len(featured):,} | Churn rate: {featured[TARGET].mean():.1%}')
featured.head()

00:50:08 | INFO    | data | Using cached raw data at /Users/tantheta/Customer Churn Prediction & Retention Analytics/data/raw/telco_churn.csv


00:50:08 | INFO    | preprocess | Filled 11 missing TotalCharges (tenure==0 customers).


00:50:08 | INFO    | preprocess | Cleaned data: 7043 rows, 21 columns.


00:50:08 | INFO    | features | Added engineered features -> total columns: 30


Rows: 7,043 | Churn rate: 26.5%


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,tenure_group,num_services,has_addon_service,avg_charges_per_tenure,charge_tenure_ratio,is_premium,is_autopay,is_month_to_month,has_family
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,0,0-6m,1,1,29.850000,14.925000,0,0,1,1
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,0,2-4y,2,1,55.573529,1.627143,0,0,0,0
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,1,0-6m,2,1,54.075000,17.950000,0,0,1,0
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,0,2-4y,3,1,40.905556,0.919565,0,1,0,0
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,1,0-6m,0,0,75.825000,23.566667,0,0,1,0


## 2. Statistical EDA
We don't just eyeball charts — we attach significance. Chi-square + **Cramér's V** rank categorical drivers by *effect size* (not just p-value), and Welch t-tests + **Cohen's d** do the same for numeric features.

In [3]:
from src.analysis.eda import run_eda
eda = run_eda(featured, TARGET)
print('Top categorical associations (by Cramer\'s V):')
display(eda['categorical_association'].head(8))
print('Numeric significance (by |Cohen\'s d|):')
display(eda['numeric_significance'].head(8))

00:50:08 | INFO    | eda | Computed categorical associations for 17 features.


00:50:08 | INFO    | eda | Computed numeric significance for 12 features.


Top categorical associations (by Cramer's V):


,feature,chi2,p_value,cramers_v,significant_5pct
13,Contract,1184.596572,5.863038e-258,0.409798,True
16,tenure_group,927.329275,1.995934e-199,0.362101,True
7,OnlineSecurity,849.998968,2.661150e-185,0.347016,True
10,TechSupport,828.197068,1.443084e-180,0.342526,True
6,InternetService,732.309590,9.571788e-160,0.322037,True
15,PaymentMethod,648.142327,3.682355e-140,0.302677,True
8,OnlineBackup,601.812790,2.079759e-131,0.291850,True
9,DeviceProtection,558.419369,5.505219e-122,0.281095,True


Numeric significance (by |Cohen's d|):


,feature,mean_churned,mean_retained,t_stat,p_value,cohens_d,significant_5pct
10,is_month_to_month,0.885500,0.429068,45.275165,0.000000e+00,1.096641,True
1,tenure,17.979133,37.569965,-34.823819,1.195495e-232,-0.892829,True
7,charge_tenure_ratio,11.745880,3.612215,28.072868,2.189360e-148,0.861060,True
9,is_autopay,0.262172,0.497874,-19.125452,7.452631e-78,-0.500483,True
3,TotalCharges,1531.796094,2549.911442,-18.706618,5.902581e-75,-0.479840,True
2,MonthlyCharges,74.441332,61.265124,18.407527,8.592449e-73,0.469507,True
6,avg_charges_per_tenure,74.433154,61.181674,18.341982,2.749968e-72,0.468681,True
11,has_family,0.399144,0.583108,-13.891280,1.090604e-42,-0.374309,True


## 3. Model bake-off + calibration
We compare Logistic Regression, Random Forest, and XGBoost with **PR-AUC** (the honest metric under class imbalance), then calibrate the winner so probabilities are trustworthy for the dollar calculations that follow.

In [4]:
from src.models.train import run_training
training = run_training(config)
pd.DataFrame(training['report']['leaderboard'])

00:50:08 | INFO    | data | Using cached raw data at /Users/tantheta/Customer Churn Prediction & Retention Analytics/data/raw/telco_churn.csv


00:50:08 | INFO    | preprocess | Filled 11 missing TotalCharges (tenure==0 customers).


00:50:08 | INFO    | preprocess | Cleaned data: 7043 rows, 21 columns.


00:50:08 | INFO    | preprocess | Saved processed data to /Users/tantheta/Customer Churn Prediction & Retention Analytics/data/processed/telco_processed.csv


00:50:08 | INFO    | features | Added engineered features -> total columns: 30


...

00:50:15 | INFO    | evaluate | [logistic_regression | test @0.50] PR-AUC=0.669 | ROC-AUC=0.848 | F1=0.579 | Recall=0.505 | Precision=0.677 | Acc=0.805 | Brier=0.134 (thr=0.50)


00:50:15 | INFO    | evaluate | [logistic_regression | test @0.22] PR-AUC=0.669 | ROC-AUC=0.848 | F1=0.616 | Recall=0.832 | Precision=0.490 | Acc=0.725 | Brier=0.134 (thr=0.22)


00:50:15 | INFO    | train | Saved calibrated model to /Users/tantheta/Customer Churn Prediction & Retention Analytics/models/churn_model.joblib


00:50:15 | INFO    | train | Saved metrics report to /Users/tantheta/Customer Churn Prediction & Retention Analytics/reports/metrics_report.json


,model,cv_score,val_pr_auc,val_roc_auc
0,logistic_regression,0.672270,0.649506,0.838545
1,xgboost,0.659957,0.632703,0.830321
2,random_forest,0.647699,0.605655,0.821074


In [5]:
training['report']['test_metrics_optimal']

{'threshold': 0.22,
 'accuracy': 0.7253371185237757,
 'precision': 0.48976377952755906,
 'recall': 0.8315508021390374,
 'f1': 0.6164519326065411,
 'roc_auc': 0.8483802216538789,
 'pr_auc': 0.6687501314107807,
 'brier': 0.13449122576343706,
 'tn': 711,
 'fp': 324,
 'fn': 63,
 'tp': 311,
 'specificity': 0.6869565217391305}

## 4. From probability to profit
Default 0.5 is rarely optimal. Using customer value V, offer cost C, and success rate s, incremental campaign profit is `TP*(s*V - C) - FP*C`. We sweep thresholds to maximise it; the rule reduces to **contact when P(churn) > C/(s*V)**.

In [6]:
training['report']['threshold_optimization']

{'best_threshold': 0.22,
 'best_profit': 62605.83413958398,
 'baseline_threshold': 0.5,
 'baseline_profit': 43630.20678072786,
 'treat_all_profit': 37559.22465213163,
 'profit_uplift_vs_default': 18975.627358856123,
 'customer_value': 1088.228383708838,
 'retention_offer_cost': 60.0,
 'offer_success_rate': 0.3}

## 5. Survival analysis — *when* do they leave?
Kaplan–Meier curves by contract type and a Cox proportional-hazards model give hazard ratios for each driver.

In [7]:
from src.analysis import survival as surv
median_surv = surv.median_survival_by_group(featured, 'Contract', TARGET)
display(median_surv)
cox = surv.fit_cox_model(featured, TARGET)
display(cox)

,Contract,median_survival_months,n_customers,churn_rate
0,Month-to-month,35.0,3875,0.427097
1,One year,inf,1473,0.112695
2,Two year,inf,1695,0.028319


00:50:15 | INFO    | survival | Fitted Cox PH model on 7 features.


,feature,coef,hazard_ratio,p_value
0,is_month_to_month,1.441103,4.225353,2.870323e-197
1,SeniorCitizen,0.118220,1.125492,1.483367e-02
2,MonthlyCharges,0.007780,1.007810,1.780808e-19
3,is_premium,-0.062488,0.939425,2.287310e-01
4,num_services,-0.184436,0.831573,2.623124e-39
5,has_family,-0.469731,0.625170,6.551670e-31
6,is_autopay,-0.581360,0.559138,5.224111e-42


## 6. CLV & Expected Value at Risk
We weight each customer's churn probability by their lifetime value to rank by **dollars at risk** — the list a retention team should actually work.

In [8]:
from src.analysis import clv as clv_mod
feature_cols = training['data']['feature_cols']
full_proba = training['calibrated'].predict_proba(featured[feature_cols])[:, 1]
var_df = clv_mod.value_at_risk(featured, full_proba, config)
var_df[['customerID','MonthlyCharges','churn_probability','clv','expected_value_at_risk']].head(10)

00:50:15 | INFO    | clv | Total expected value at risk: $1406301. Top 10% of customers hold $421878 (30%) of that risk.


,customerID,MonthlyCharges,churn_probability,clv,expected_value_at_risk
3837,3932-CMDTD,105.65,0.583348,1768.789673,1031.820250
6482,5419-JPRRN,101.45,1.000000,889.676531,889.676531
2208,7216-EWTRS,100.80,1.000000,883.976287,883.976287
6894,1400-MMYXY,105.90,0.947949,928.701278,880.361186
171,1875-QIVME,104.40,0.949048,915.546869,868.897576
2246,7181-BQYBV,102.45,0.963333,898.446137,865.503112
5933,6496-SLWHQ,105.00,0.933663,920.808632,859.724953
4459,3178-FESZO,100.25,0.963333,879.153004,846.917394
1704,0107-YHINA,99.75,0.955003,874.768201,835.405993
2797,6023-YEBUP,100.95,0.942419,885.291728,834.316031


## 7. Segmentation & uplift targeting
K-means segments the base into actionable groups (value vs risk), and a proxy uplift score separates **persuadables** from sure-things and lost-causes.

In [9]:
from src.analysis import segmentation as seg, uplift as up
segmented = seg.segment_customers(featured, config)
clv_series = clv_mod.compute_clv(segmented, config)
strategy = seg.build_strategy_matrix(segmented, full_proba, clv_series, TARGET)
display(strategy)
uplift_df = up.uplift_targeting(featured, full_proba, budget_fraction=0.2)
uplift_df['persuadable_category'].value_counts()

00:50:15 | INFO    | segmentation | Segmented 7043 customers into 4 clusters.


00:50:15 | INFO    | segmentation | Built retention strategy matrix for 4 segments.


,segment,n_customers,avg_tenure,avg_monthly_charges,avg_num_services,avg_churn_prob,avg_clv,actual_churn_rate,segment_label,recommended_strategy,total_value_at_risk
3,3,2373,15.052676,77.579499,1.831858,0.483238,767.447137,0.473662,High-value / High-risk,Protect: concierge outreach + targeted loyalty...,880049.134458
1,1,2158,56.890176,90.594393,4.159870,0.153688,1855.525141,0.154310,High-value / Low-risk,"Nurture: upsell add-ons, preserve satisfaction",615399.725218
0,0,1521,10.159763,29.580473,0.297830,0.222424,341.526306,0.241946,Low-value / High-risk,Automate: low-cost digital retention nudges,115540.655927
2,2,991,54.538850,31.812159,0.581231,0.042766,671.479617,0.044400,Low-value / Low-risk,Monitor: minimal proactive spend,28458.038871


00:50:15 | INFO    | uplift | Uplift targeting: 1408 customers flagged within 20% budget; 3085 persuadables identified.


persuadable_category
Persuadable (target!)              3085
Low priority                       3077
Lost cause (don't waste budget)     444
Sure thing (already staying)        437
Name: count, dtype: int64

## Takeaways
- A **calibrated, explainable** model beats a black box you can't defend to stakeholders.
- Optimising the **decision threshold** for profit — not accuracy — is where the business value is.
- Ranking by **value at risk** and targeting **persuadables** focuses limited retention budget where it pays off.

Run `python main.py` to regenerate all figures/tables, and `streamlit run dashboard/app.py` for the interactive app.